In [0]:
"""
aggregations.py
================
Business logic for the Gold profit summary table: profit by year,
product category, product sub-category, and customer.
"""
import logging
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType, DateType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from setup import load_config, get_all_tables,get_raw_path
from src.utils.schemas import ORDERS_SCHEMA,CUSTOMER_SCHEMA,PRODUCTS_SCHEMA,GOLD_SALES_SCHEMA
import pyspark.sql.functions as F
 
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
 
logger = logging.getLogger(__name__)


In [0]:
config = load_config()

# unpack into a dict — same names as your original constants
tables = get_all_tables(config)
path=get_raw_path(config)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("Aggregation layer")
 
GOLD_SALES       = tables["GOLD_SALES"]
ENRICHED_ORDERS =  tables["GOLD_ENRICHED"]

In [0]:
ENRICHED_ORDERS

In [0]:
def write_gold(df: DataFrame, target: str,schema = GOLD_SALES_SCHEMA) -> None:
    """Upsert into silver Delta table; create on first run."""
    df = df.select(*[
        F.col(f.name).cast(f.dataType).alias(f.name) for f in schema.fields
    ])
    spark.catalog.tableExists(target)      # warm up catalog
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("year").saveAsTable(target)

In [0]:
def optimize_table(table: str, zorder_cols: list = None):
    sql = f"OPTIMIZE {table}"
    if zorder_cols:
        sql += f" ZORDER BY ({', '.join(zorder_cols)})"
    spark.sql(sql)
    logger.info(f"  OPTIMIZE complete: {table}")

In [0]:
def build_profit_summary() -> DataFrame:
    """Aggregate profit to (year, category, sub_category, customer).
    """
    order_by = ["year", "category", "sub_category", "customer_id"]
    enriched_orders= spark.table(ENRICHED_ORDERS)
    enriched_orders = enriched_orders.withColumn("year", F.year("order_date"))
    enriched_orders_agg= enriched_orders.groupBy(*order_by).agg(
        F.sum("profit").alias("total_profit"),
        F.count(F.lit(1)).alias("order_count"))
    write_gold(enriched_orders_agg, GOLD_SALES,schema=GOLD_SALES_SCHEMA)
    optimize_table(GOLD_SALES, zorder_cols=["category", "sub_category", "customer_id"])
    print(f"Table load completed for table {GOLD_SALES}")

    return enriched_orders
    

In [0]:
if __name__ == "__main__":
    build_profit_summary()
    print("\nGold layer build complete.")